# Embeddings in Recommender systems

Learning about recommender systems and embeddings using keras and pytorch


Following the separate plans generated with AI. We will start with Keras (easier code) and then do pytorch.

AI use: The goal here is to use AI as a teaching tool, but I will write all the code myself to reinforce the concepts and execution.

In [1]:
# Load libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import os
from sklearn.model_selection import train_test_split

# Phase 1
## Load the data
Import the data, and save it to a folder for later reloading

In [2]:
# Get the data saved to files
ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/refs/heads/master/ratings.csv"
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/refs/heads/master/books.csv"

local_dir = "data"
ratings_path = os.path.join(local_dir,"ratings.csv")
books_path = os.path.join(local_dir,"books.csv")

os.makedirs(local_dir,exist_ok=True)

if not os.path.exists(ratings_path) and not os.path.exists(books_path):
    ratings = pd.read_csv(ratings_url)
    ratings.to_csv(ratings_path,index=False)

    books = pd.read_csv(books_url)
    books.to_csv(books_path,index=False)
else:
    ratings = pd.read_csv(ratings_path)
    books = pd.read_csv(books_path)

## Merge and clean IDs

In [3]:
ratings['user_idx'] = ratings['user_id'].astype('category').cat.codes
ratings['book_idx'] = ratings['book_id'].astype('category').cat.codes

ratings.head()

,user_id,book_id,rating,user_idx,book_idx
0,1,258,5,0,257
1,2,4081,4,1,4080
2,2,260,5,1,259
3,2,9296,5,1,9295
4,2,2318,3,1,2317


In [ ]:
# Create dictionaries to map between original ids and the new 0-indexed idx's
book_id_to_idx = dict(zip(ratings['book_id'],ratings['book_idx']))
idx_to_book_id = {v: k for k, v in book_id_to_idx.items()}

user_id_to_idx = dict(zip(ratings['user_id'],ratings['user_idx']))
idx_to_user_id = {v: k for k, v in user_id_to_idx.items()}

# Phase 2: Data pipeline

## Train/validation/test split

In [ ]:
# 80% training, 10% validation, 10% testing
train_df, temp_df = train_test_split(ratings, test_size=0.2, random_state=2026)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=2026)

## Create datasets

In [24]:
# simple function to crease tf datasets from the dataframes
def create_tf_dataset(df, batch_size=2048, shuffle=True):
    # create slices (?)
    ds = tf.data.Dataset.from_tensor_slices((
        # a dictionary of the user and book indices as input, and the rating as output
        {'user_id': df['user_idx'].values, 'book_id': df['book_idx'].values},
         df['rating'].values))
    if shuffle:
        # shuffle the dataset (buffer size = length of the dataframe)
        ds = ds.shuffle(buffer_size=len(df))
    # batch the dataset, cache it, and prefetch it for performance
    ds = ds.batch(batch_size).cache().prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_tf_dataset(train_df)
val_ds = create_tf_dataset(val_df, shuffle=False)
test_ds = create_tf_dataset(test_df, shuffle=False)

# Phase 3: Define Model
*   **Input Layers:** Create two `tf.keras.layers.Input` layers, one for `user_input` and one for `book_input`.
*   **Embedding Layers:** 
    *   Create `tf.keras.layers.Embedding` layers for users and books. Apply `tf.keras.regularizers.l2(1e-5)` directly within the embedding layer definition to prevent overfitting.
    *   Create bias embedding layers (output dimension = 1) for users and books.
*   **Dot Product & Addition:** 
    *   Use `tf.keras.layers.Dot(axes=1)` to compute the interaction between the user and book embeddings.
    *   Use `tf.keras.layers.Add()` to add the dot product and the two bias terms together.
*   **Model Compilation:** 
    *   Define the model: `model = tf.keras.Model(inputs=[user_input, book_input], outputs=prediction)`.

In [25]:
embedding_dim = 32

# user input is just grabbing the user_id column 
user_input = tf.keras.Input(shape=(), dtype=tf.int32, name='user_id')
# embedding with a +1 for unknown users, output dimension = embedding_dim, and L2 regularization
user_embedding = tf.keras.layers.Embedding(
    input_dim=len(user_id_to_idx)+1,
    output_dim = embedding_dim,
    embeddings_regularizer = tf.keras.regularizers.l2(1e-5),
    name = 'user_embedding')(user_input) # calling the layer on the user_input to get the actual embeddings

# bias embedding with output dimension = 1 and L2 regularization
user_bias = tf.keras.layers.Embedding(
    input_dim = len(user_id_to_idx) + 1,
    output_dim = 1,
    embeddings_regularizer = tf.keras.regularizers.l2(1e-5),
    name = 'user_bias')(user_input)

# same here for books
book_input = tf.keras.Input(shape=(), dtype=tf.int32, name='book_id')
book_embedding = tf.keras.layers.Embedding(
    input_dim = len(book_id_to_idx) + 1,
    output_dim = embedding_dim,
    embeddings_regularizer = tf.keras.regularizers.l2(1e-5),
    name = 'book_embedding')(book_input)
book_bias = tf.keras.layers.Embedding(
    input_dim = len(book_id_to_idx) + 1,
    output_dim = 1,
    embeddings_regularizer = tf.keras.regularizers.l2(1e-5),
    name = 'book_bias')(book_input)

# target is the dot product of the user and book embeddings, plus the user and book bias terms
score = tf.keras.layers.Dot(axes=1)([user_embedding, book_embedding])
score = tf.keras.layers.Add()([score, user_bias, book_bias])

# construct model with inputs and outputs
model = tf.keras.Model(inputs=[user_input, book_input], outputs = score)

# how model should be trained
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss = tf.keras.losses.MeanSquaredError()
)

# Phase 4: Training
*   **Compilation:** `model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')`.
*   **Callbacks:**
    *   Use `tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)` to automatically handle overfitting.
*   **Execution:** Call `model.fit(train_dataset, validation_data=val_dataset, epochs=20, callbacks=[early_stopping])`. This single line replaces the entire custom loop required in PyTorch.

In [26]:
# fit the model. this may be slow...
model.fit(
    train_ds,
    validation_data = val_ds,
    epochs = 10,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
)

Epoch 1/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 6.3936 - val_loss: 2.2318
Epoch 2/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 2.1600 - val_loss: 2.1198
Epoch 3/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 2.0784 - val_loss: 2.0488
Epoch 4/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 18s 8ms/step - loss: 2.0133 - val_loss: 1.9935
Epoch 5/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 1.9634 - val_loss: 1.9517
Epoch 6/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 1.9252 - val_loss: 1.9199
Epoch 7/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 1.8955 - val_loss: 1.8954
Epoch 8/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - loss: 1.8724 - val_loss: 1.8765
Epoch 9/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - loss: 1.8542 - val_loss: 1.8618
Epoch 10/10
2335/2335 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - loss: 1.8396 - val_loss: 1.8502


## Phase 5: Extraction, Visualization, and Analysis
*   **Extract Weights:** Access the weights directly from the layer: `book_embeddings = model.get_layer('book_embedding').get_weights()[0]`.
*   **Dimensionality Reduction:** Use Scikit-learn's `TSNE` or `UMAP` to map the arrays to 2D.
*   **Visualization:** Create an interactive scatter plot with Plotly.
*   **Vector Math:** Use Scikit-learn's `cosine_similarity` module to find the nearest neighbors for a target book embedding.

In [28]:
trained_book_embeddings = model.get_layer('book_embedding').get_weights()[0]